Author: Krish with additions from Ella and Liv

In [1]:
# Importing relevant libraries
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from datetime import datetime
import numpy as np
import warnings
warnings.filterwarnings("ignore")
from datetime import date

# Reading data

In [3]:
data_path = "/Users/ellammarks/Desktop/CAR_-_EP_Flow_Activity_Queue__Agent_Names"

In [4]:
adhoc_data_path = "/Users/ellammarks/Desktop/Adhoc datasets"

In [5]:
folder = Path(data_path)
files = sorted(list(folder.glob("*.csv")) + list(folder.glob("*.xlsx")))
df_main = pd.DataFrame(columns=['Contact Session ID', 'EP Name', 'Flow Name', 'Activity Name', 'Activity Start Timestamp', 
                                'Queue Name', 'Agent Name', 'Termination Reason'])
i=0
for f in files:
    i = i + 1
    if f.suffix.lower() == ".csv":
        df = pd.read_csv(f, header=2, dtype=str, engine="python", skip_blank_lines=False)
    else:  # .xlsx
        df = pd.read_excel(f, sheet_name=0, header=2, dtype=str)
    df_main = pd.concat([df_main, df], ignore_index=True)
print("Data files read = ",i)

Data files read =  53


In [6]:
df_main["Activity Start Timestamp"] = df_main["Activity Start Timestamp"].apply(
    lambda x: datetime.strptime(x, "%Y/%m/%d %I:%M:%S %p"))

In [7]:
def custdata(id):
    return df_main.loc[df_main['Contact Session ID'] == id,:]

In [8]:
df_main.shape

(3328626, 8)

In [9]:
df_main.dtypes

Contact Session ID                  object
EP Name                             object
Flow Name                           object
Activity Name                       object
Activity Start Timestamp    datetime64[ns]
Queue Name                          object
Agent Name                          object
Termination Reason                  object
dtype: object

In [10]:
df_main["hour"] = df_main["Activity Start Timestamp"].dt.hour

In [11]:
df_main.sort_values(by = ['Contact Session ID', 'Activity Start Timestamp'], inplace=True)

In [12]:
df_main.reset_index(inplace = True, drop = True)

In [13]:
df_main['Date'] = df_main['Activity Start Timestamp'].dt.date

In [97]:
df_main.to_csv(adhoc_data_path + 'df_main.csv')

In [98]:
# determining Contact Session IDs with no activities
empty_activity_ids = (
    df_main.groupby("Contact Session ID")["Activity Name"]
      .apply(lambda x: x.isna().all() or (x.str.strip() == "").all())
      .loc[lambda x: x]
      .index
)

print(empty_activity_ids.tolist())

['00906e65-550d-4190-9add-da5a8d3a452e', '03d6ebbc-1aa0-450b-92bb-33e6ec1055d0', '04d6296e-d77b-4c90-80dd-b194a8e653a7', '04d7e372-6ff6-41b4-813e-bc18dd6dabfe', '077fc423-c9e5-4429-a299-a9d37e23fdc8', '07ccff31-418a-4eac-b3f1-84aeac8339b0', '09be3dab-c9ed-4f0d-a123-b21942cd430e', '0b6be61e-4c87-44db-b25f-14b5f2f22e77', '0eaa8dfa-1589-4a41-abdd-68517c648054', '0fe864b3-18d2-4ff1-bf1b-1531003b48e8', '13963227-78e8-4d97-809e-3294d30b6ac8', '13c12267-ed3b-4248-b7d5-57b061f3173a', '16c3fa95-db0c-4905-b740-5752a80af68c', '18ba633e-4506-42a9-8e58-503147ba111f', '18fa370c-a8ff-4e04-acd8-b3a33648707e', '19c6b4bb-157a-49ef-a9fd-cc0e737e6d1d', '1a7d2bcb-fbdd-48d1-85ec-bedcab8bbc18', '1b6c41a8-514a-4996-8c19-55b9d7a47372', '1ba7dd42-7b14-4e04-9207-6d0f57a6b6fb', '1bfbe049-d30a-4d1c-8b55-61da5a447200', '1c158e4c-46e6-4c0b-a4c8-ef18f62ba293', '1cad4fd7-c3c9-42d6-a88d-3d5495749cee', '1d5c6abd-297d-448b-bea4-f2d9b30607b6', '1df09a52-37e1-44fa-b18a-665c8b1f2cc3', '1f3cbc83-f2e1-468a-81fd-929928bc0c6f',

# Data prep for dashboard
## Data for the Menu Traffic and Hourly trends tabs of the dashboard (link below)
https://app.powerbi.com/groups/me/reports/01ca1c28-e56b-42fe-89c9-4cd53882a4c7/c1da62005af463baaa15?experience=power-bi

In [43]:
# Next activity and the one after that (per session), given rows are already sorted
def dash_data(ep_name, activity_name):
    familymenu_rows = df_main.loc[df_main['EP Name'] == ep_name,:]
    familymenu_data = df_main.loc[df_main['Contact Session ID'].isin(familymenu_rows['Contact Session ID']),:]
    familymenu_activity = familymenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    familymenu_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df2 = familymenu_activity.copy()
    
    g = df2.groupby("Contact Session ID")
    df2["Second Activity"]      = g["Activity Name"].shift(-1)
    df2["Third Activity"]  = g["Activity Name"].shift(-2)
    df2["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df2["Fifth Activity"]  = g["Activity Name"].shift(-4)
    
    out = (
        df2.loc[df2["Activity Name"].eq(activity_name),
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    # select the 3rd, 4th, 5th, and 6th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 8]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                  .str.replace("FamilySP", "Family SP", regex=False)
                  .str.replace("EducationSP", "Education SP", regex=False)
                  .str.replace("ConsumerSP", "Consumer SP", regex=False)
                  .str.replace("BenefitsSP", "Benefits SP", regex=False)
                  .str.replace("EmploymentSP", "Employment SP", regex=False)
                  .str.replace("ImmigrationSP", "Immigration SP", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("GetLoggedInFamilySPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInHousingAgents", "Housing Queue", regex=False)
                  .str.replace("GetLoggedInHousingSPAgents", "Housing SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTAgents", "ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSPAgents", "ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumerAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsSPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefitsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsSPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilyAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilySPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsSPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("ADAPTSPQueue", "ADAPT SP Queue", regex=False)
                  .str.replace("ADAPTQueue", "ADAPT Queue", regex=False)
                  .str.replace("EmploymentQueue", "Employment Queue", regex=False)
                  .str.replace("ImmigrationQueue", "Immigration Queue", regex=False)
                  .str.replace("BenefitsQueue", "Benefits Queue", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("EducationQueue", "Education Queue", regex=False)
                  .str.replace("ConsumerQueue", "Consumer Queue", regex=False)
                  .str.replace("TenantDeterrenceMenu", "Tenant Deterrence", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("HIVVoicemailTransfer", "HIV Voicemail Transfer", regex=False)
                  .str.replace("ClinicVoicemailTransfer", "Clinic Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("TenantMenu", "Tenant", regex=False)
                  .str.replace("MainMenu", "Main", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
                  .str.replace("IntakePreQueueMessage1", "Intake Pre Queue Message", regex=False)
                  .str.replace("OtherLegalOtherMenu", "Other Legal", regex=False)
                  .str.replace("OtherLegalPersonalInjuryMenu", "Other Legal Personal Injury", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("CriminalRecordsVoicemailTransfer", "Criminal Records Voicemail Transfer", regex=False)
                  .str.replace("LegalMenu1", "Legal 1", regex=False)
                  .str.replace("LegalMenu2", "Legal 2", regex=False)
                  .str.replace("SeniorsADAPTMenu", "Seniors ADAPT", regex=False)
                  .str.replace("FrontDeskTransfer", "Front Desk Transfer", regex=False)
                  .str.replace("TransferToSafeHaven", "Transfer to Safe Haven", regex=False)
                  .str.replace("SeniorsConfirmationMenu", "Seniors Confirmation", regex=False)
                  .str.replace("SeniorsMenu", "Seniors", regex=False)
                  .str.replace("SuburbanSeniors", "Suburban Seniors", regex=False)
                  .str.replace("SuburbsOrCityMenu", "Suburbs or City", regex=False)
                  .str.replace("FarmworkerMainMenu", "Farmworker Main", regex=False)
                  .str.replace("LanguageSelectionMenu", "Language Selection", regex=False)
                  .str.replace("ClosedMenu", "Closed Menu", regex=False)
                  .str.replace("MigrantVoicemailTransfer", "Migrant Voicemail Transfer", regex=False)
                  .str.replace("AddressFaxHoursMenu", "Address Fax Hours", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("StaffDirectoryEnglishTransfer", "Staff Directory English Transfer", regex=False)
                  .str.replace("StaffDirectorySpanishTransfer", "Staff Directory Spanish Transfer", regex=False)
                  .str.replace("PreTenantMenu", "PreTenant", regex=False)
                  .str.replace("SeniorNotCookCoMenu", "Senior Not Cook County", regex=False)
                  .str.replace("ThankYouGoodbye", "Thank You Goodbye", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
                  .str.replace("PreQueueMessage2", "Pre Queue Message 2", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("LegalServerScreenPop", "Legal Server Screen Pop", regex=False)
        )
    return out

In [42]:
out_family = dash_data('Legal Family Menu Telephony EP', 'FamilyMenu')
out_housing = dash_data('Legal Housing Menu Telephony EP', 'HousingMenu')
out_benefits = dash_data('Legal Benefits Menu Telephony EP', 'BenefitsMenu')
out_hiv = dash_data('Legal HIV Menu Telephony EP', 'HIVMenu')
out_imm = dash_data('Legal Immigration Menu Telephony EP', 'ImmigrationMenu')
out_emp = dash_data('Legal Employment Menu Telephony EP', 'EmploymentMenu')

out_all = pd.concat([out_family, out_housing, out_benefits, out_hiv, out_imm, out_emp]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'family_dash.csv', index = False)

In [34]:
# Pre-Legal Seniors Menu
def dash_data2(ep_name, activity_name):
    prelegalseniorsmenu_rows = df_main.loc[df_main['EP Name'] == 'Pre-Legal Menu Seniors Menu Telephony EP',:]
    prelegalseniorsmenu_data = df_main.loc[df_main['Contact Session ID'].isin(prelegalseniorsmenu_rows['Contact Session ID']),:]
    prelegalseniorsmenu_activity = prelegalseniorsmenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    prelegalseniorsmenu_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df3 = prelegalseniorsmenu_activity.copy()
    
    g = df3.groupby("Contact Session ID")
    df3["Second Activity"]      = g["Activity Name"].shift(-1)
    df3["Third Activity"]  = g["Activity Name"].shift(-2)
    df3["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df3["Fifth Activity"]  = g["Activity Name"].shift(-4)
    
    out = (
        df3.loc[df3["Activity Name"].eq(activity_name),
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    # select the 3rd, 4th, 5th and 6th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 8]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                  .str.replace("FamilySP", "Family SP", regex=False)
                  .str.replace("EducationSP", "Education SP", regex=False)
                  .str.replace("ConsumerSP", "Consumer SP", regex=False)
                  .str.replace("BenefitsSP", "Benefits SP", regex=False)
                  .str.replace("EmploymentSP", "Employment SP", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("GetLoggedInFamilySPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTAgents", "ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSPAgents", "ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumerAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsSPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefitsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsSPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilyAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilySPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsSPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("HIVVoicemailTransfer", "HIV Voicemail Transfer", regex=False)
                  .str.replace("ClinicVoicemailTransfer", "Clinic Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("TenantMenu", "Tenant", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
                  .str.replace("IntakePreQueueMessage1", "Intake Pre Queue Message", regex=False)
                  .str.replace("OtherLegalOtherMenu", "Other Legal", regex=False)
                  .str.replace("OtherLegalPersonalInjuryMenu", "Other Legal Personal Injury", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("CriminalRecordsVoicemailTransfer", "Criminal Records Voicemail Transfer", regex=False)
                  .str.replace("LegalMenu1", "Legal 1", regex=False)
                  .str.replace("LegalMenu2", "Legal 2", regex=False)
                  .str.replace("SeniorsADAPTMenu", "Seniors ADAPT", regex=False)
                  .str.replace("FrontDeskTransfer", "Front Desk Transfer", regex=False)
                  .str.replace("TransferToSafeHaven", "Transfer to Safe Haven", regex=False)
                  .str.replace("SeniorsConfirmationMenu", "Seniors Confirmation", regex=False)
                  .str.replace("SeniorsMenu", "Seniors", regex=False)
                  .str.replace("SuburbanSeniors", "Suburban Seniors", regex=False)
                  .str.replace("SuburbsOrCityMenu", "Suburbs or City", regex=False)
                  .str.replace("FarmworkerMainMenu", "Farmworker Main", regex=False)
                  .str.replace("LanguageSelectionMenu", "Language Selection", regex=False)
                  .str.replace("ClosedMenu", "Closed Menu", regex=False)
                  .str.replace("MigrantVoicemailTransfer", "Migrant Voicemail Transfer", regex=False)
                  .str.replace("AddressFaxHoursMenu", "Address Fax Hours", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("StaffDirectoryEnglishTransfer", "Staff Directory English Transfer", regex=False)
                  .str.replace("StaffDirectorySpanishTransfer", "Staff Directory Spanish Transfer", regex=False)
                  .str.replace("PreTenantMenu", "PreTenant", regex=False)
                  .str.replace("SeniorNotCookCoMenu", "Senior Not Cook County", regex=False)
                  .str.replace("ThankYouGoodbye", "Thank You Goodbye", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
        )
    return out

In [35]:
out_seniors = dash_data2('Pre-Legal Menu Seniors Menu Telephony EP', 'SeniorsMenu')

out_all = pd.concat([out_seniors]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'prelegalseniors_dash.csv', index = False)

In [85]:
# Farmworker Menu
def dash_data3(ep_name, activity_name):
    farmworkermenu_rows = df_main.loc[df_main['EP Name'] == 'Farmworker Main Number Telephony EP',:]
    farmworkermenu_data = df_main.loc[df_main['Contact Session ID'].isin(farmworkermenu_rows['Contact Session ID']),:]
    farmworkermenu_activity = farmworkermenu_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    farmworkermenu_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df4 = farmworkermenu_activity.copy()

    g = df4.groupby("Contact Session ID")

    df4["Activity Position"] = g.cumcount() #new
    
    g = df4.groupby("Contact Session ID")
    df4["Second Activity"]      = g["Activity Name"].shift(-1)
    df4["Third Activity"]  = g["Activity Name"].shift(-2)
    df4["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df4["Fifth Activity"]  = g["Activity Name"].shift(-4)
    df4["Sixth Activity"]  = g["Activity Name"].shift(-5)
    df4["Seventh Activity"]  = g["Activity Name"].shift(-6)
    df4["Eighth Activity"]  = g["Activity Name"].shift(-7)
    df4["Ninth Activity"]  = g["Activity Name"].shift(-8)
    df4["Tenth Activity"]  = g["Activity Name"].shift(-9)
    df4["Eleventh Activity"]  = g["Activity Name"].shift(-10)
    df4["Twelfth Activity"]  = g["Activity Name"].shift(-11)
    df4["Thirteenth Activity"]  = g["Activity Name"].shift(-12)
    df4["Fourteenth Activity"]  = g["Activity Name"].shift(-13)
    df4["Fifteenth Activity"]  = g["Activity Name"].shift(-14)
    df4["Sixteenth Activity"]  = g["Activity Name"].shift(-15)
    df4["Seventeenth Activity"]  = g["Activity Name"].shift(-16)
    df4["Eighteenth Activity"]  = g["Activity Name"].shift(-17)
    df4["Nineteenth Activity"]  = g["Activity Name"].shift(-18)
    df4["Twentieth Activity"]  = g["Activity Name"].shift(-19)
    df4["Twentyfirst Activity"]  = g["Activity Name"].shift(-20)
    df4["Twentysecond Activity"]  = g["Activity Name"].shift(-21)
    
    out = (
        df4.loc[(df4["Activity Name"] == activity_name) & (df4["Activity Position"] == 0), # added df4["Activity Position"] == 0
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity", "Sixth Activity", "Seventh Activity", "Eighth Activity", "Ninth Activity", "Tenth Activity", "Eleventh Activity", "Twelfth Activity", "Thirteenth Activity", "Fourteenth Activity", "Fifteenth Activity", "Sixteenth Activity", "Seventeenth Activity", "Eighteenth Activity", "Nineteenth Activity", "Twentieth Activity", "Twentyfirst Activity", "Twentysecond Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    # select the 3rd through 6th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 25]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                   .str.replace("FamilySP", "Family SP", regex=False)
                  .str.replace("EducationSP", "Education SP", regex=False)
                  .str.replace("ConsumerSP", "Consumer SP", regex=False)
                  .str.replace("BenefitsSP", "Benefits SP", regex=False)
                  .str.replace("EmploymentSP", "Employment SP", regex=False)
                  .str.replace("ImmigrationSP", "Immigration SP", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefits SPAgents", "Benefits SP Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInEducation SPAgents", "Education SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInFamily SPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumer SPAgents", "Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInImmigration SPAgents", "Immigration SP Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmployment SPAgents", "Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilySPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInHousingAgents", "Housing Queue", regex=False)
                  .str.replace("GetLoggedInHousingSPAgents", "Housing SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTAgents", "ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSPAgents", "ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumerAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumer SPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsSPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefitsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsSPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefits SPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilyAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilySPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamily SPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsSPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmployment SPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsSPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorTenantAgents", "Sub Senior Tenant Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorTenantSPAgents", "Sub Senior Tenant SP Queue", regex=False)
                  .str.replace("GetLoggedInHousingSubSeniorsAgents", "Sub Senior Housing Queue", regex=False)
                  .str.replace("ADAPTSPQueue", "ADAPT SP Queue", regex=False)
                  .str.replace("ADAPTQueue", "ADAPT Queue", regex=False)
                  .str.replace("EmploymentQueue", "Employment Queue", regex=False)
                  .str.replace("ImmigrationQueue", "Immigration Queue", regex=False)
                  .str.replace("BenefitsQueue", "Benefits Queue", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("EducationQueue", "Education Queue", regex=False)
                  .str.replace("Consumer SPQueue", "Consumer SP Queue", regex=False)
                  .str.replace("ConsumerQueue", "Consumer Queue", regex=False)
                  .str.replace("TenantDeterrenceMenu", "Tenant Deterrence", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("HIVVoicemailTransfer", "HIV Voicemail Transfer", regex=False)
                  .str.replace("ClinicVoicemailTransfer", "Clinic Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("TenantMenu", "Tenant", regex=False)
                  .str.replace("MainMenu", "Main", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
                  .str.replace("IntakePreQueueMessage1", "Intake Pre Queue Message", regex=False)
                  .str.replace("OtherLegalOtherMenu", "Other Legal", regex=False)
                  .str.replace("OtherLegalPersonalInjuryMenu", "Other Legal Personal Injury", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("CriminalRecordsVoicemailTransfer", "Criminal Records Voicemail Transfer", regex=False)
                  .str.replace("LegalMenu1", "Legal 1", regex=False)
                  .str.replace("LegalMenu2", "Legal 2", regex=False)
                  .str.replace("SeniorsADAPTMenu", "Seniors ADAPT", regex=False)
                  .str.replace("FrontDeskTransfer", "Front Desk Transfer", regex=False)
                  .str.replace("TransferToSafeHaven", "Transfer to Safe Haven", regex=False)
                  .str.replace("SeniorsConfirmationMenu", "Seniors Confirmation", regex=False)
                  .str.replace("SeniorsMenu", "Seniors", regex=False)
                  .str.replace("SuburbanSeniors", "Suburban Seniors", regex=False)
                  .str.replace("SuburbsOrCityMenu", "Suburbs or City", regex=False)
                  .str.replace("FarmworkerMainMenu", "Farmworker Main", regex=False)
                  .str.replace("ClosedMenu", "Closed Menu", regex=False)
                  .str.replace("MigrantVoicemailTransfer", "Migrant Voicemail Transfer", regex=False)
                  .str.replace("AddressFaxHoursMenu", "Address Fax Hours", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("DisconnectContact2", "Disconnect Contact 2", regex=False)
                  .str.replace("StaffDirectoryEnglishTransfer", "Staff Directory English Transfer", regex=False)
                  .str.replace("StaffDirectorySpanishTransfer", "Staff Directory Spanish Transfer", regex=False)
                  .str.replace("PreTenantMenu", "PreTenant", regex=False)
                  .str.replace("SeniorNotCookCoMenu", "Senior Not Cook County", regex=False)
                  .str.replace("ThankYouGoodbye", "Thank You Goodbye", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
                  .str.replace("PreQueueMessage2", "Pre Queue Message 2", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("LegalServerScreenPop", "Legal Server Screen Pop", regex=False)
                  .str.replace("HelpWithLegalorOtherReasonMenu", "Help with Legal or Other Reason", regex=False)
                  .str.replace("ComplimentOrComplaintMenu", "Compliment or Complaint", regex=False)
                  .str.replace("AppointmentMenu", "Appointment", regex=False)
                  .str.replace("PlayErrorMessage", "Play Error Message", regex=False)
                  .str.replace("SubSeniorPreQueueMessage1_1", "Sub Senior Pre Queue Message 1-1", regex=False)
                  .str.replace("SubSeniorPreQueueMessage1_2", "Sub Senior Pre Queue Message 1-2", regex=False)
                  .str.replace("QueueMenu1", "Queue Menu 1", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("PlayMOH300s", "Play MOH 300s", regex=False)
                  .str.replace("LegalServerScreenPop", "Legal Server Screen Pop", regex=False)
                  .str.replace("CallbackRetry", "Callback Retry", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue Menu", regex=False)
                  .str.replace("LegalMenu2", "Legal Menu 2", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("PlayCCBConfirmation", "Play CCB Confirmation", regex=False)
                  .str.replace("DisconnectCallbackContact", "Disconnect Callback Contact", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
                  .str.replace("CollectCallbackNumber", "Collect Callback Number", regex=False)
                  .str.replace("ConfirmCallbackNumber", "Confirm Callback Number", regex=False)
                  .str.replace("ScreenPopProcessComplete", "Screen Pop Process Complete", regex=False)
                  .str.replace("TenantDeterrenceMenu", "Tenant Deterrence Menu", regex=False)
                  .str.replace("DisconnectCallbackContact", "Disconnect Callback Contact", regex=False)
                  .str.replace("LanguageSelectionMenu", "Language Selection Farmworker", regex=False)
                  .str.replace("FarmworkerMain", "Farmworker Main", regex=False)
        )
    return out

In [86]:
# create csv for only Farmworker Menu
out_language = dash_data3('Farmworker Main Number Telephony EP', 'LanguageSelectionMenu')

out_all = pd.concat([out_language]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixth Activity'].isna(),'Sixth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventh Activity'].isna(),'Seventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighth Activity'].isna(),'Eighth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Ninth Activity'].isna(),'Ninth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Tenth Activity'].isna(),'Tenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eleventh Activity'].isna(),'Eleventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twelfth Activity'].isna(),'Twelfth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Thirteenth Activity'].isna(),'Thirteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourteenth Activity'].isna(),'Fourteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifteenth Activity'].isna(),'Fifteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixteenth Activity'].isna(),'Sixteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventeenth Activity'].isna(),'Seventeenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighteenth Activity'].isna(),'Eighteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Nineteenth Activity'].isna(),'Nineteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentieth Activity'].isna(),'Twentieth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentyfirst Activity'].isna(),'Twentyfirst Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentysecond Activity'].isna(),'Twentysecond Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'farmworker_dash.csv', index = False)

In [99]:
# LAC Main
def dash_data4(ep_name, activity_name):
    lacmain_rows = df_main.loc[df_main['EP Name'] == ep_name,:]
    lacmain_data = df_main.loc[df_main['Contact Session ID'].isin(lacmain_rows['Contact Session ID']),:]
    lacmain_activity = lacmain_data[['Contact Session ID', 'Date', 'hour', 'Activity Name']]
    lacmain_activity.dropna(subset=["Activity Name"], inplace=True)
    
    df5 = lacmain_activity.copy()
    
    g = df5.groupby("Contact Session ID")

    df5["Activity Position"] = g.cumcount() #new
    
    df5["Second Activity"]      = g["Activity Name"].shift(-1)
    df5["Third Activity"]  = g["Activity Name"].shift(-2)
    df5["Fourth Activity"]  = g["Activity Name"].shift(-3)
    df5["Fifth Activity"]  = g["Activity Name"].shift(-4)
    df5["Sixth Activity"]  = g["Activity Name"].shift(-5)
    df5["Seventh Activity"]  = g["Activity Name"].shift(-6)
    df5["Eighth Activity"]  = g["Activity Name"].shift(-7)
    df5["Ninth Activity"]  = g["Activity Name"].shift(-8)
    df5["Tenth Activity"]  = g["Activity Name"].shift(-9)
    df5["Eleventh Activity"]  = g["Activity Name"].shift(-10)
    df5["Twelfth Activity"]  = g["Activity Name"].shift(-11)
    df5["Thirteenth Activity"]  = g["Activity Name"].shift(-12)
    df5["Fourteenth Activity"]  = g["Activity Name"].shift(-13)
    df5["Fifteenth Activity"]  = g["Activity Name"].shift(-14)
    df5["Sixteenth Activity"]  = g["Activity Name"].shift(-15)
    df5["Seventeenth Activity"]  = g["Activity Name"].shift(-16)
    df5["Eighteenth Activity"]  = g["Activity Name"].shift(-17)
    df5["Nineteenth Activity"]  = g["Activity Name"].shift(-18)
    df5["Twentieth Activity"]  = g["Activity Name"].shift(-19)
    df5["Twentyfirst Activity"]  = g["Activity Name"].shift(-20)
    df5["Twentysecond Activity"]  = g["Activity Name"].shift(-21)
    
    out = (
        df5.loc[(df5["Activity Name"] == activity_name) & (df5["Activity Position"] == 0), # added df5["Activity Position"] == 0
                ["Contact Session ID", 'Date', 'hour', "Activity Name", "Second Activity", "Third Activity", "Fourth Activity", "Fifth Activity", "Sixth Activity", "Seventh Activity", "Eighth Activity", "Ninth Activity", "Tenth Activity", "Eleventh Activity", "Twelfth Activity", "Thirteenth Activity", "Fourteenth Activity", "Fifteenth Activity", "Sixteenth Activity", "Seventeenth Activity", "Eighteenth Activity", "Nineteenth Activity", "Twentieth Activity", "Twentyfirst Activity", "Twentysecond Activity"]]
           .rename(columns={"Activity Name": "Current Activity"})
           .reset_index(drop=True)
    )
    
    # select the 3rd through 9th columns by position (0-based index → 2 and 3)
    cols = out.columns[3: 25]
    
    # do substring replacements, preserving NA values
    for c in cols:
        out[c] = (
            out[c].astype("string")  # use pandas StringDtype to keep <NA>
                  .str.replace("FamilySP", "Family SP", regex=False)
                  .str.replace("EducationSP", "Education SP", regex=False)
                  .str.replace("ConsumerSP", "Consumer SP", regex=False)
                  .str.replace("BenefitsSP", "Benefits SP", regex=False)
                  .str.replace("EmploymentSP", "Employment SP", regex=False)
                  .str.replace("ImmigrationSP", "Immigration SP", regex=False)
                  .str.replace("GetLoggedInBenefitsAgents", "Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefits SPAgents", "Benefits SP Queue", regex=False)
                  .str.replace("VeteransBenefitsVoicemailTransfer", "Veterans Benefits Voicemail Transfer", regex=False)
                  .str.replace("GetLoggedInEducationAgents", "Education Queue", regex=False)
                  .str.replace("GetLoggedInEducation SPAgents", "Education SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilyAgents", "Family Queue", regex=False)
                  .str.replace("GetLoggedInFamily SPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInConsumerAgents", "Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumer SPAgents", "Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInImmigrationAgents", "Immigration Queue", regex=False)
                  .str.replace("GetLoggedInImmigration SPAgents", "Immigration SP Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentAgents", "Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmployment SPAgents", "Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilySPAgents", "Family SP Queue", regex=False)
                  .str.replace("GetLoggedInHousingAgents", "Housing Queue", regex=False)
                  .str.replace("GetLoggedInHousingSPAgents", "Housing SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTAgents", "ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSPAgents", "ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsAgents", "Sub Senior Other Queue", regex=False)
                  .str.replace("GetLoggedInOtherSubSeniorsSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumerAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsAgents", "Sub Senior Consumer Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorConsumer SPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInConsumerSubSeniorsSPAgents", "Sub Senior Consumer SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefitsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsAgents", "Sub Senior Benefits Queue", regex=False)
                  .str.replace("GetLoggedInBenefitsSubSeniorsSPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorBenefits SPAgents", "Sub Senior Benefits SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilyAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsAgents", "Sub Senior Family Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamilySPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorFamily SPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInFamilySubSeniorsSPAgents", "Sub Senior Family SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorOtherSPAgents", "Sub Senior Other SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsAgents", "Sub Senior ADAPT Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorADAPTSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInADAPTSubSeniorsSPAgents", "Sub Senior ADAPT SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmploymentAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsAgents", "Sub Senior Employment Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorEmployment SPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInEmploymentSubSeniorsSPAgents", "Sub Senior Employment SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsSPAgents", "Sub Senior Homeowner SP Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorHomeownerAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInHomeownerSubSeniorsAgents", "Sub Senior Homeowner Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorTenantAgents", "Sub Senior Tenant Queue", regex=False)
                  .str.replace("GetLoggedInSubSeniorTenantSPAgents", "Sub Senior Tenant SP Queue", regex=False)
                  .str.replace("GetLoggedInHousingSubSeniorsAgents", "Sub Senior Housing Queue", regex=False)
                  .str.replace("ADAPTSPQueue", "ADAPT SP Queue", regex=False)
                  .str.replace("ADAPTQueue", "ADAPT Queue", regex=False)
                  .str.replace("EmploymentQueue", "Employment Queue", regex=False)
                  .str.replace("ImmigrationQueue", "Immigration Queue", regex=False)
                  .str.replace("BenefitsQueue", "Benefits Queue", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("EducationQueue", "Education Queue", regex=False)
                  .str.replace("Consumer SPQueue", "Consumer SP Queue", regex=False)
                  .str.replace("ConsumerQueue", "Consumer Queue", regex=False)
                  .str.replace("TenantDeterrenceMenu", "Tenant Deterrence", regex=False)
                  .str.replace("WorkersCompMenu", "Workers Compensation", regex=False)
                  .str.replace("ImmigrationOtherMenu", "Immigration Other Menu", regex=False)
                  .str.replace("BenefitsMenu", "Benefits", regex=False)
                  .str.replace("FamilyMenu", "Family", regex=False)
                  .str.replace("HIVMenu", "HIV", regex=False)
                  .str.replace("HousingMenu", "Housing", regex=False)
                  .str.replace("ImmigrationMenu", "Immigration", regex=False)
                  .str.replace("TraffickingVoicemailTransfer", "Trafficking Voicemail Transfer", regex=False)
                  .str.replace("HIVVoicemailTransfer", "HIV Voicemail Transfer", regex=False)
                  .str.replace("ClinicVoicemailTransfer", "Clinic Voicemail Transfer", regex=False)
                  .str.replace("EmploymentMenu", "Employment", regex=False)
                  .str.replace("TenantMenu", "Tenant", regex=False)
                  .str.replace("MainMenu", "Main", regex=False)
                  .str.replace("HolidayPrompt", "Holiday Prompt", regex=False)
                  .str.replace("DivorceOrParentingMenu", "Divorce / Parenting", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue", regex=False)
                  .str.replace("ChildSupportMenu", "Child Support", regex=False)
                  .str.replace("SimpleDivorceMenu", "Simple Divorce", regex=False)
                  .str.replace("TransferToSafeHaven", "Safe Haven", regex=False)
                  .str.replace("IntakePreQueueMessage1", "Intake Pre Queue Message", regex=False)
                  .str.replace("OtherLegalOtherMenu", "Other Legal", regex=False)
                  .str.replace("OtherLegalPersonalInjuryMenu", "Other Legal Personal Injury", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("CriminalRecordsVoicemailTransfer", "Criminal Records Voicemail Transfer", regex=False)
                  .str.replace("LegalMenu1", "Legal 1", regex=False)
                  .str.replace("LegalMenu2", "Legal 2", regex=False)
                  .str.replace("SeniorsADAPTMenu", "Seniors ADAPT", regex=False)
                  .str.replace("FrontDeskTransfer", "Front Desk Transfer", regex=False)
                  .str.replace("TransferToSafeHaven", "Transfer to Safe Haven", regex=False)
                  .str.replace("SeniorsConfirmationMenu", "Seniors Confirmation", regex=False)
                  .str.replace("SeniorsMenu", "Seniors", regex=False)
                  .str.replace("SuburbanSeniors", "Suburban Seniors", regex=False)
                  .str.replace("SuburbsOrCityMenu", "Suburbs or City", regex=False)
                  .str.replace("FarmworkerMainMenu", "Farmworker Main", regex=False)
                  .str.replace("LanguageSelectionMenu", "Language Selection", regex=False)
                  .str.replace("ClosedMenu", "Closed Menu", regex=False)
                  .str.replace("MigrantVoicemailTransfer", "Migrant Voicemail Transfer", regex=False)
                  .str.replace("AddressFaxHoursMenu", "Address Fax Hours", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("DisconnectContact2", "Disconnect Contact 2", regex=False)
                  .str.replace("StaffDirectoryEnglishTransfer", "Staff Directory English Transfer", regex=False)
                  .str.replace("StaffDirectorySpanishTransfer", "Staff Directory Spanish Transfer", regex=False)
                  .str.replace("PreTenantMenu", "PreTenant", regex=False)
                  .str.replace("SeniorNotCookCoMenu", "Senior Not Cook County", regex=False)
                  .str.replace("ThankYouGoodbye", "Thank You Goodbye", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
                  .str.replace("PreQueueMessage2", "Pre Queue Message 2", regex=False)
                  .str.replace("FamilyQueue", "Family Queue", regex=False)
                  .str.replace("LegalServerScreenPop", "Legal Server Screen Pop", regex=False)
                  .str.replace("HelpWithLegalorOtherReasonMenu", "Help with Legal or Other Reason", regex=False)
                  .str.replace("ComplimentOrComplaintMenu", "Compliment or Complaint", regex=False)
                  .str.replace("AppointmentMenu", "Appointment", regex=False)
                  .str.replace("PlayErrorMessage", "Play Error Message", regex=False)
                  .str.replace("SubSeniorPreQueueMessage1_1", "Sub Senior Pre Queue Message 1-1", regex=False)
                  .str.replace("SubSeniorPreQueueMessage1_2", "Sub Senior Pre Queue Message 1-2", regex=False)
                  .str.replace("QueueMenu1", "Queue Menu 1", regex=False)
                  .str.replace("DisconnectContact1", "Disconnect Contact 1", regex=False)
                  .str.replace("PlayMOH300s", "Play MOH 300s", regex=False)
                  .str.replace("LegalServerScreenPop", "Legal Server Screen Pop", regex=False)
                  .str.replace("CallbackRetry", "Callback Retry", regex=False)
                  .str.replace("ClosedQueueMenu", "Closed Queue Menu", regex=False)
                  .str.replace("LegalMenu2", "Legal Menu 2", regex=False)
                  .str.replace("OtherLegalMenu", "Other Legal", regex=False)
                  .str.replace("PlayCCBConfirmation", "Play CCB Confirmation", regex=False)
                  .str.replace("DisconnectCallbackContact", "Disconnect Callback Contact", regex=False)
                  .str.replace("OtherLegalCriminalCaseMenu", "Other Legal Criminal Case", regex=False)
                  .str.replace("CollectCallbackNumber", "Collect Callback Number", regex=False)
                  .str.replace("ConfirmCallbackNumber", "Confirm Callback Number", regex=False)
                  .str.replace("ScreenPopProcessComplete", "Screen Pop Process Complete", regex=False)
                  .str.replace("TenantDeterrenceMenu", "Tenant Deterrence Menu", regex=False)
                  .str.replace("DisconnectCallbackContact", "Disconnect Callback Contact", regex=False)
                  .str.replace("N/A", "Abandoned", regex=False)
        )
    return out

In [83]:
# create csv for only LAC Main Menu
out_language = dash_data4('Main Number Telephony EP', 'LanguageSelectionMenu')
out_seniors = dash_data4('Main Number Telephony EP', 'SeniorsMenu')
out_abandoned = dash_data4('Main Number Telephony EP', 'N/A')

out_all = pd.concat([out_language, out_seniors, out_abandoned]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixth Activity'].isna(),'Sixth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventh Activity'].isna(),'Seventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighth Activity'].isna(),'Eighth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Ninth Activity'].isna(),'Ninth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Tenth Activity'].isna(),'Tenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eleventh Activity'].isna(),'Eleventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twelfth Activity'].isna(),'Twelfth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Thirteenth Activity'].isna(),'Thirteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourteenth Activity'].isna(),'Fourteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifteenth Activity'].isna(),'Fifteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixteenth Activity'].isna(),'Sixteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventeenth Activity'].isna(),'Seventeenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighteenth Activity'].isna(),'Eighteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Nineteenth Activity'].isna(),'Nineteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentieth Activity'].isna(),'Twentieth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentyfirst Activity'].isna(),'Twentyfirst Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentysecond Activity'].isna(),'Twentysecond Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'lacmain6_dash.csv', index = False)

In [101]:
# create csv for combined Farmworker and LAC Main data

# Define all activity columns
activity_cols = ["Current Activity", "Second Activity", "Third Activity", "Fourth Activity", 
                 "Fifth Activity", "Sixth Activity", "Seventh Activity", "Eighth Activity",
                 "Ninth Activity", "Tenth Activity", "Eleventh Activity", "Twelfth Activity",
                 "Thirteenth Activity", "Fourteenth Activity", "Fifteenth Activity", "Sixteenth Activity",
                 "Seventeenth Activity", "Eighteenth Activity", "Nineteenth Activity", "Twentieth Activity",
                 "Twentyfirst Activity", "Twentysecond Activity"]

# Create the DataFrame for empty activity IDs
df_empty_activities = pd.DataFrame([
    {**{"Contact Session ID": csid}, **{col: "Abandoned / End of call" for col in activity_cols}}
    for csid in empty_activity_ids
])

out_language = dash_data4('Main Number Telephony EP', 'LanguageSelectionMenu')
out_seniors = dash_data4('Main Number Telephony EP', 'SeniorsMenu')
out_language2 = dash_data3('Farmworker Main Number Telephony EP', 'LanguageSelectionMenu')

out_all = pd.concat([out_language, out_seniors, df_empty_activities, out_language2]).reset_index(drop = True)
out_all.loc[out_all['Second Activity'].isna(),'Second Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Third Activity'].isna(),'Third Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourth Activity'].isna(),'Fourth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifth Activity'].isna(),'Fifth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixth Activity'].isna(),'Sixth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventh Activity'].isna(),'Seventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighth Activity'].isna(),'Eighth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Ninth Activity'].isna(),'Ninth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Tenth Activity'].isna(),'Tenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eleventh Activity'].isna(),'Eleventh Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twelfth Activity'].isna(),'Twelfth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Thirteenth Activity'].isna(),'Thirteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fourteenth Activity'].isna(),'Fourteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Fifteenth Activity'].isna(),'Fifteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Sixteenth Activity'].isna(),'Sixteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Seventeenth Activity'].isna(),'Seventeenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Eighteenth Activity'].isna(),'Eighteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Nineteenth Activity'].isna(),'Nineteenth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentieth Activity'].isna(),'Twentieth Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentyfirst Activity'].isna(),'Twentyfirst Activity'] = 'Abandoned / End of call'
out_all.loc[out_all['Twentysecond Activity'].isna(),'Twentysecond Activity'] = 'Abandoned / End of call'
out_all.to_csv(adhoc_data_path + 'combined_dash.csv', index = False)